## What is Whoosh for Wii U?

Whoosh for Wii U is a port of Whoosh, a social platform inspired by earlier versions of Twitter. It is also a standalone space for text-based conversations, where people can share interests ranging from music, films, books, and fashion to personal collections and niche hobbies.

The Wii U version adds features designed for the console:

- **GamePad posts:** Write posts by hand on the Wii U GamePad and decorate them with stamps earned in games or unlocked through Whoosh.
- **SpotPass notifications:** Receive notifications in the Wii U Notifications applet and access exclusive content.
- **Mii and friend connections:** Use your Mii and connect with Nintendo Network revival services such as Pretendo Network to see whether your Wii U friends are on Whoosh, or whether your Whoosh friends are on Wii U.

Availability of network features may depend on the services and integrations supported by the project.

## APIs and programming languages required

This section expands on the features described above and distinguishes what the repository already demonstrates from what a complete, networked Wii U application would still need. The current `website/` directory is an HTML/CSS/JavaScript prototype; `app/main.cpp` is the native C++ entry point, and the VS Code configuration selects a PowerPC EABI C++ compiler and Wii U headers/tooling. There is not yet a backend service or a documented Whoosh API in this repository. Consequently, the service routes below are proposed API contracts, not existing endpoints.

### Languages and formats

| Language or format | Role in this project | Required? |
|---|---|---|
| **C++ (configured as C++17)** | Native Wii U application: system integration, controller/GamePad input, rendering, audio, local settings, and network-client code. The repository's editor configuration uses `powerpc-eabi-g++`; it also points at Wii U toolchain headers. | Required for the native application as currently scaffolded. Confirm the language standard against the actual compiler/toolchain before relying on every C++17 feature. |
| **HTML** | Structure and accessible controls for the web prototype, or web-based screens if a supported embedded/browser runtime is chosen. | Required for the existing web prototype, not intrinsically required for a wholly native C++ UI. |
| **CSS** | Layout, colors, responsive presentation, typography, icon fonts, and visual themes in the web prototype. | Required for the existing web prototype; native screens instead need the native renderer/UI framework. |
| **JavaScript** | Interactions and browser-side behavior in the prototype: DOM events, focus/navigation, the browser Gamepad API, and audio elements. | Required for the existing web prototype, not necessarily for the native client. |
| **JSON** | Interchange format for requests and responses between a client and the proposed service. It is a data format, not a programming language. | Recommended for the network API. |
| **SQL** | Persistent relational storage for accounts, posts, relationships, messages, and other server-side data, if a relational database is selected. | Backend implementation choice; no database is present in the repository. |
| **Backend language** | Implements the Whoosh service, authentication, validation, database access, and integrations. | A service is needed for real shared social features, but no language or server framework has been selected. It could be TypeScript/JavaScript, Python, Go, or another maintained server language. |

The web prototype and native application are two different client approaches, not a requirement to put all four client languages in one program. A sensible implementation can retain the web prototype for design/testing and build the Wii U release in C++. The backend is a separate component. HTTP, HTTPS, REST, and TCP/IP are protocols or architectural styles, not programming languages.

### APIs already used by the web prototype

- **DOM and HTML element APIs:** `document.getElementById`, `getElementsByTagName`, `getElementsByClassName`, `activeElement`, element attributes, `focus()`, and `click()` update controls and move focus.
- **Browser event APIs:** `addEventListener` handles keyboard and page-load events; keyboard events implement arrow-key focus movement.
- **Gamepad API:** `navigator.getGamepads()` (with a legacy `webkitGetGamepads`/`webkitGamepads` fallback) polls controller axes and buttons. This is browser support, not the Wii U native controller API; compatibility must be tested on the actual target runtime.
- **Timers and audio:** `window.setInterval` polls the controller and `new Audio(...)` creates an HTML audio element. The current `Pressed()` function constructs an audio object but does not play it, so sound playback still needs to be completed.
- **CSS/font loading:** stylesheets and local fonts provide presentation and symbols. These are assets/resources rather than remote service APIs.

These browser APIs do not currently call a Whoosh server. In particular, the visible search screen is only a UI, and the menu entries marked as not implemented do not yet fetch feeds, search results, or account data.

### Native Wii U APIs and libraries

The native client needs the Wii U development toolchain and its supported system libraries (the repository configuration references `devkitPro`, `devkitPPC`, Wii U headers, and the `__WIIU__` build define). Use the documented APIs available in the chosen Wii U SDK/homebrew environment for:

- **Application lifecycle and graphics:** initialize and tear down the application, draw the TV and GamePad interfaces, handle screen changes, and present user feedback. Choose one supported graphics/UI approach and keep rendering separate from network and business logic.
- **Controller and GamePad input:** read buttons, sticks, touch coordinates, and stylus input; map those to navigation, text entry, and hand-drawn posts. The browser Gamepad API is not a substitute for this native platform interface.
- **Text entry and handwriting:** accept on-screen keyboard input and, if handwritten posts are a real requirement, capture touch strokes, render them, and serialize/upload the resulting image or stroke data. This needs an explicit data format, size limit, and moderation/abuse policy.
- **Audio:** play interface and notification sounds, honor the user's sound setting, manage audio resources, and handle missing/unsupported assets cleanly.
- **Networking and TLS:** connect to the configured service over authenticated HTTPS, send/receive JSON and media, enforce timeouts, and report offline/server errors. Select an HTTP/TLS client supported by the target toolchain rather than assuming desktop browser facilities exist on Wii U.
- **Local storage:** persist appropriate preferences (for example theme and sound setting) using platform-supported storage. Keep secrets out of plain-text settings where possible, and provide logout/credential removal.
- **Mii display and identity:** use supported Mii-related APIs only for the permitted purpose, request consent where needed, and avoid sending a console or network identifier unless necessary and explicitly authorized. Treat the Mii as profile/display data, not proof of authentication.
- **Notifications / SpotPass:** this is conditional platform integration, not a generic web API. Verify that the selected development environment, network service, account state, and applicable platform rules actually permit the intended delivery path. If not, use an in-app notification inbox and do not promise SpotPass delivery.

The precise names and availability of platform functions depend on the SDK/runtime used to build the executable. The repository's configuration indicates intended toolchain paths, but it is not evidence that every listed platform integration is implemented or available to a homebrew application.

### Proposed Whoosh web-service API

For multiple users and devices to share content, the project needs a server-side API, a database, and an authentication design. A versioned HTTPS JSON REST API is a straightforward starting point. The following route names illustrate capabilities that must be designed; they are **not implemented or confirmed endpoints**. The backend should publish an OpenAPI (or equivalent) contract before clients depend on it.

| Capability | Example operations the service must define | Purpose |
|---|---|---|
| Service health and configuration | `GET /v1/health`, `GET /v1/config` | Check service availability and return non-secret client configuration, supported features, and API version. |
| Account and authentication | `POST /v1/auth/register`, `POST /v1/auth/login`, `POST /v1/auth/refresh`, `POST /v1/auth/logout` | Create/login to Whoosh accounts, issue/refresh/revoke scoped sessions, and support recovery if the service chooses to offer accounts. Do not send or store passwords in logs. |
| Current user and profile | `GET /v1/me`, `GET /v1/users/{userId}`, `PATCH /v1/me` | Read/update a profile, with strict field allow-lists and privacy settings. |
| Posts | `POST /v1/posts`, `GET /v1/posts/{postId}`, `PATCH /v1/posts/{postId}`, `DELETE /v1/posts/{postId}` | Create, read, edit, and delete posts, checking ownership and content limits on the server. |
| Feeds | `GET /v1/feed/home`, `GET /v1/feed/following`, `GET /v1/feed/trending`, `GET /v1/users/{userId}/posts` | Load home/following, popular, recent, and profile feeds. Define stable ordering, pagination cursors, filtering, and empty-feed behavior. |
| Social graph | `GET /v1/users/{userId}/followers`, `GET /v1/users/{userId}/following`, `PUT/DELETE /v1/users/{userId}/follow` | Read and change follow relationships, respecting blocks, privacy, and authorization. |
| Reactions | `PUT/DELETE /v1/posts/{postId}/reactions/{reaction}` | Add/remove supported reactions without duplicate state changes. |
| Search | `GET /v1/search?q=...` | Search users/posts with server-side validation, pagination, privacy filtering, and rate limiting. The prototype search input does not yet implement this call. |
| Direct messages | `GET /v1/conversations`, `POST /v1/conversations`, `GET /v1/conversations/{id}/messages`, `POST /v1/conversations/{id}/messages` | Optional private messaging. Decide access controls, retention, reporting/blocking, and notification behavior before enabling it. |
| Media | `POST /v1/media` or a documented signed-upload flow; `GET /v1/media/{mediaId}` | Upload/serve images or handwritten-post exports with size/type checks, safe filenames, quotas, moderation, and authorization. Avoid trusting client-provided MIME types. |
| Notifications | `GET /v1/notifications`, `PATCH /v1/notifications/{id}` | Provide an in-app inbox, unread state, and notification preferences; platform push/SpotPass delivery is a separate optional integration. |
| Preferences and themes | `GET/PATCH /v1/me/settings`, `GET /v1/themes` | Sync user-level preferences or theme catalog where appropriate. Device-only preferences such as sound may remain local. |
| Nintendo Network revival/friend integration | A separately documented integration boundary, if the provider permits one | Cross-check friends only through supported, authorized interfaces. Do not assume Pretendo exposes a public Whoosh API or call private/undocumented endpoints. Provide a consent-based fallback if integration is unavailable. |

The server should return consistent JSON objects, documented HTTP status codes, and machine-readable error codes; validate every field and authorization check on the server even when the client validates too. Use cursor pagination rather than downloading an unlimited feed. Define request timeouts, retry behavior for safe/idempotent operations, rate limits, and a versioning/deprecation policy. Avoid putting access tokens or personal content in URLs because URLs are commonly logged.

### Data and security decisions

At minimum, the service will need a design for users/profiles, posts, follow relationships, reactions, and preferences. Add messages, uploaded media, notifications, and external-network links only when those features are in scope. Define unique IDs, timestamps, visibility, deletion/retention behavior, and indexes for feed/search queries. Use parameterized database queries and migrations; do not let the client connect directly to the database.

Use TLS for network traffic and protect passwords with a modern password-hashing scheme on the server. Keep signing keys, database credentials, and privileged service tokens on the server, never in the Wii U executable or website JavaScript. Use short-lived, revocable client sessions; minimize console/account identifiers; validate external integration responses; and provide clear user consent, block/report controls, and data deletion processes. Since a distributed client can be inspected or modified, client-side checks cannot enforce authorization or protect embedded secrets.

### Practical implementation order

1. Decide whether the deliverable is a browser prototype, a native Wii U client, or both; document the supported toolchain and runtime.
2. Specify the service API and data model for accounts, profiles, posts, and feeds; build a small backend and test it independently.
3. Connect one client path end-to-end using HTTPS/JSON, beginning with login (if accounts are required), profile, create-post, and paginated feed operations.
4. Add follows, search, themes/settings, media, and messages as separate capabilities with server-side authorization and tests.
5. Implement native GamePad input, text/handwriting, audio, local preferences, and Mii display using only APIs supported by the chosen Wii U environment.
6. Treat SpotPass and Pretendo/Nintendo Network friend discovery as optional integrations until their supported interfaces, authorization, and actual target-device behavior have been verified; retain useful in-app alternatives.